# Convert Recovered ECG `.npy` Files to `.mat` Files

**What this notebook does:**
- Walks the `RECOVERED_DIR` folder tree (mirrors the PTB-XL structure: `recovered_ecg/00000/00001_hr.npy`)
- Loads each `.npy` file — shape `(N, num_leads)`, dtype `float32`
- Saves a `.mat` file in the same relative location under `MAT_SAVE_DIR`
- The `.mat` file contains:
  - `ecg`         — `(N, num_leads)` float32 matrix
  - `lead_names`  — cell array of lead name strings
  - `record_name` — string identifier (e.g. `'00001_hr'`)

```
Input  : recovered_ecg/00000/00001_hr.npy   shape (5000, 12)
Output : recovered_mat/00000/00001_hr.mat
```

## 1. Imports & Configuration

In [1]:
import os
import numpy as np
import scipy.io as sio
from pathlib import Path
from tqdm import tqdm   # pip install tqdm  (progress bar)

# ── Paths ──────────────────────────────────────────────────────────────────
# Directory where Stage-1 saved the recovered .npy files
RECOVERED_DIR = "/Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3"

# Directory where this notebook will write .mat files
# Folder structure is mirrored: recovered_mat/00000/00001_hr.mat
MAT_SAVE_DIR  = "/Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3"

# Lead order that Stage-1 used when building the recovered matrix
# Must match the `leads` argument passed to recover_and_save()
ALL_LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

os.makedirs(MAT_SAVE_DIR, exist_ok=True)
print(f"Input  (recovered .npy): {RECOVERED_DIR}")
print(f"Output (converted .mat): {MAT_SAVE_DIR}")
print(f"Lead order             : {ALL_LEADS}")

Input  (recovered .npy): /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3
Output (converted .mat): /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3
Lead order             : ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


## 2. Helper — Load `.npy` and Save `.mat`

In [2]:
def convert_npy_to_mat(npy_path: str,
                       mat_path: str,
                       lead_names: list = ALL_LEADS) -> bool:
    """
    Load a recovered .npy file and save it as a .mat file.

    Args:
        npy_path   : full path to the source .npy file
        mat_path   : full path for the output .mat file
        lead_names : ordered list of lead name strings

    Returns:
        True on success, False on any error.

    .mat contents:
        ecg         — (N, num_leads) float32 array
        lead_names  — (1, num_leads) cell array of strings   [MATLAB-friendly]
        record_name — char array, e.g. '00001_hr'
        fs          — scalar 500 (PTB-XL high-res sampling rate)
    """
    try:
        ecg = np.load(npy_path)               # shape (N, num_leads)
    except Exception as e:
        print(f"  [LOAD ERROR] {npy_path}: {e}")
        return False

    # Sanity-check lead count
    if ecg.shape[1] != len(lead_names):
        print(f"  [SHAPE MISMATCH] {npy_path}: "
              f"expected {len(lead_names)} leads, got {ecg.shape[1]}")
        # Still save — caller can decide what to do

    record_name = Path(npy_path).stem   # e.g. '00001_hr'

    # scipy.io.savemat expects a cell array as a numpy object array
    lead_names_cell = np.array(lead_names, dtype=object).reshape(1, -1)

    mat_dict = {
        'ecg'         : ecg,
        'lead_names'  : lead_names_cell,
        'record_name' : record_name,
        'fs'          : 500,          # PTB-XL high-resolution sampling rate
    }

    os.makedirs(os.path.dirname(mat_path), exist_ok=True)
    try:
        sio.savemat(mat_path, mat_dict)
    except Exception as e:
        print(f"  [SAVE ERROR] {mat_path}: {e}")
        return False

    return True

print("convert_npy_to_mat() defined.")

convert_npy_to_mat() defined.


## 3. Discover All `.npy` Files

In [3]:
npy_files = sorted(Path(RECOVERED_DIR).rglob('*.npy'))
print(f"Found {len(npy_files):,} .npy files under {RECOVERED_DIR}")

# Preview first few
for p in npy_files[:5]:
    print(' ', p)

Found 194 .npy files under /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3/00000/00001_hr.npy
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3/00000/00002_hr.npy
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3/00000/00003_hr.npy
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3/00000/00004_hr.npy
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3/00000/00005_hr.npy


## 4. Single-File Test

In [4]:
if len(npy_files) == 0:
    print("No .npy files found — check RECOVERED_DIR path.")
else:
    # Pick the first file as a test
    test_npy = str(npy_files[0])

    # Mirror the subfolder path under MAT_SAVE_DIR
    rel_path  = os.path.relpath(test_npy, RECOVERED_DIR)
    test_mat  = os.path.join(MAT_SAVE_DIR, Path(rel_path).with_suffix('.mat'))

    print(f"Source : {test_npy}")
    print(f"Dest   : {test_mat}")

    ok = convert_npy_to_mat(test_npy, test_mat)

    if ok:
        # Verify round-trip
        loaded = sio.loadmat(test_mat, squeeze_me=True)
        print(f"\n.mat keys       : {list(loaded.keys())}")
        print(f"ecg shape       : {loaded['ecg'].shape}")
        print(f"ecg dtype       : {loaded['ecg'].dtype}")
        print(f"lead_names      : {loaded['lead_names']}")
        print(f"record_name     : {loaded['record_name']}")
        print(f"fs              : {loaded['fs']}")
        print("\n✅ Single-file test passed.")
    else:
        print("❌ Test failed — see error above.")

Source : /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_ecg3/00000/00001_hr.npy
Dest   : /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00001_hr.mat

.mat keys       : ['__header__', '__version__', '__globals__', 'ecg', 'lead_names', 'record_name', 'fs']
ecg shape       : (5000, 12)
ecg dtype       : float32
lead_names      : ['I' 'II' 'III' 'aVR' 'aVL' 'aVF' 'V1' 'V2' 'V3' 'V4' 'V5' 'V6']
record_name     : 00001_hr
fs              : 500

✅ Single-file test passed.


## 5. Batch Convert — All Files

In [5]:
n_ok    = 0
n_fail  = 0
failed  = []

for npy_path in tqdm(npy_files, desc="Converting", unit="file"):
    npy_str  = str(npy_path)

    # Build mirrored output path
    rel_path = os.path.relpath(npy_str, RECOVERED_DIR)
    mat_path = os.path.join(MAT_SAVE_DIR, str(Path(rel_path).with_suffix('.mat')))

    ok = convert_npy_to_mat(npy_str, mat_path)

    if ok:
        n_ok += 1
    else:
        n_fail += 1
        failed.append(npy_str)

print(f"\n✅ Converted : {n_ok:,}")
print(f"❌ Failed    : {n_fail:,}")
if failed:
    print("\nFailed files:")
    for f in failed:
        print(' ', f)

Converting: 100%|██████████| 194/194 [00:00<00:00, 833.14file/s]


✅ Converted : 194
❌ Failed    : 0


## 6. Verify Output Structure

In [6]:
mat_files = sorted(Path(MAT_SAVE_DIR).rglob('*.mat'))
print(f"Total .mat files written : {len(mat_files):,}")
print(f"Total .npy files found   : {len(npy_files):,}")
print()

# Preview a few output paths
print("Sample output paths:")
for p in mat_files[:5]:
    print(' ', p)

Total .mat files written : 956
Total .npy files found   : 956

Sample output paths:
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00001_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00002_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00003_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00004_hr.mat
  /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00005_hr.mat


## 7. Quick Sanity Check — Random Sample

In [7]:
import random

if len(mat_files) == 0:
    print("No .mat files found — something went wrong.")
else:
    sample_mat = str(random.choice(mat_files))
    data = sio.loadmat(sample_mat, squeeze_me=True)

    print(f"File         : {sample_mat}")
    print(f"record_name  : {data['record_name']}")
    print(f"ecg shape    : {data['ecg'].shape}   (samples × leads)")
    print(f"ecg dtype    : {data['ecg'].dtype}")
    print(f"fs           : {data['fs']} Hz")
    print(f"lead_names   : {list(data['lead_names'])}")
    print(f"Value range  : [{data['ecg'].min():.4f}, {data['ecg'].max():.4f}] mV")

File         : /Users/sinahassannia/Desktop/ecg_lead_cover2/recovered_mat3/00000/00727_hr.mat
record_name  : 00727_hr
ecg shape    : (5000, 12)   (samples × leads)
ecg dtype    : float32
fs           : 500 Hz
lead_names   : ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Value range  : [-1.5779, 2.2081] mV


## How to Load the `.mat` Files in MATLAB

```matlab
data   = load('recovered_mat/00000/00001_hr.mat');
ecg    = data.ecg;          % (5000 × 12)  double
leads  = data.lead_names;   % 1×12 cell array of strings
fs     = data.fs;           % 500

% Plot Lead I
t = (0:size(ecg,1)-1) / fs;
plot(t, ecg(:,1));
xlabel('Time (s)'); ylabel('Amplitude (mV)');
title(['Lead ' leads{1}]);
```

## How to Load in Python (scipy)

```python
import scipy.io as sio

data = sio.loadmat('recovered_mat/00000/00001_hr.mat', squeeze_me=True)
ecg        = data['ecg']          # (5000, 12) float32
lead_names = list(data['lead_names'])  # ['I', 'II', ...]
fs         = int(data['fs'])      # 500
```